## LIBRARIES AND DECLARATION

In [1]:
import os
import numpy as np 
import pandas as pd 
from IPython.display import FileLink

## CONFIG

In [2]:
ID_COL     = 'id'
INPUT_DIR  = '/kaggle/input/datasets/dhuyent/classified-filtered-stage2'

GROUPS = ['logic1', 'logic2', 'reference1']
RUNS   = ['claude-opus-4-8', 'gpt-5.6-sol'] # model       
FNAME  = 'step1_classify2_{group}_{run}.csv'

## INTERSECTION FILTERING (2 MODELS)

Filter the `id`s that neither model judged correctly.

For each group (`logic1`, `logic2`, `reference1`), take the intersection of `id`s labeled 2 or 3 across both models (`gpt-5.6-sol` and `claude-opus-4-8`):

- **Label 2**: SOME but not all attempts match (partial match)
- **Label 3**: NO attempt matches

An `id` is kept only if it falls in Label 2 or 3 for **both** models. `id`s that either model judged fully correct (Label 1) are dropped.

In [3]:
KEEP_LABELS = [2, 3]
LABEL_COL   = 'label'
# LABEL_COL   = 'filtered_label'

for group in GROUPS:
    for run in RUNS:
        path = os.path.join(INPUT_DIR, FNAME.format(group=group, run=run))
        df   = pd.read_csv(path)

        ids = df.loc[df[LABEL_COL].isin([2, 3]), ID_COL].sort_values().tolist()

        print(f'=== {group} | {run} ===')
        print(f'id (label 2/3): {len(ids)}')
        print(ids)
        print()

=== logic1 | claude-opus-4-8 ===
id (label 2/3): 19
[38, 40, 45, 60, 63, 72, 80, 91, 96, 108, 109, 110, 111, 127, 130, 132, 133, 134, 139]

=== logic1 | gpt-5.6-sol ===
id (label 2/3): 14
[38, 45, 60, 63, 72, 91, 96, 108, 109, 111, 130, 132, 133, 134]

=== logic2 | claude-opus-4-8 ===
id (label 2/3): 23
[6, 22, 23, 35, 40, 41, 44, 51, 66, 76, 82, 83, 85, 103, 111, 119, 123, 125, 128, 133, 142, 153, 158]

=== logic2 | gpt-5.6-sol ===
id (label 2/3): 19
[6, 23, 35, 41, 44, 51, 76, 82, 83, 85, 89, 103, 111, 119, 123, 133, 153, 158, 160]

=== reference1 | claude-opus-4-8 ===
id (label 2/3): 33
[18, 21, 23, 29, 35, 46, 54, 57, 61, 63, 66, 71, 72, 76, 78, 80, 81, 96, 103, 105, 107, 110, 111, 124, 125, 126, 127, 133, 139, 140, 148, 164, 172]

=== reference1 | gpt-5.6-sol ===
id (label 2/3): 28
[18, 20, 29, 35, 46, 57, 61, 63, 71, 76, 78, 80, 81, 96, 103, 105, 107, 110, 111, 112, 115, 116, 124, 133, 139, 140, 164, 172]



In [4]:
def combine_group(group):
    files = [os.path.join(INPUT_DIR, FNAME.format(group=group, run=r)) for r in RUNS]
    dfs   = [pd.read_csv(f) for f in files]

    # Lọc label trước khi giao
    if KEEP_LABELS is not None:
        dfs = [d[d[LABEL_COL].isin(KEEP_LABELS)] for d in dfs]

    id_sets = [set(d[ID_COL]) for d in dfs]
    common  = set.intersection(*id_sets)  # phép giao

    df_common = (dfs[0][dfs[0][ID_COL].isin(common)]
                 .sort_values(ID_COL).reset_index(drop=True))

    out = f'step1_filter2_{group}.csv'
    df_common.to_csv(out, index=False, encoding='utf-8-sig')

    detail = ' | '.join(f'{r}={len(s)}' for r, s in zip(RUNS, id_sets))
    print(f'[{group}] {detail} -> {len(common)} ids in intersection | Saved to {out}')
    return out

## SAVE OUTPUT

In [5]:
for g in GROUPS:
    path = combine_group(g)
    display(FileLink(path))

[logic1] claude-opus-4-8=19 | gpt-5.6-sol=14 -> 14 ids in intersection | Saved to step1_filter2_logic1.csv


/kaggle/working/step1_filter2_logic1.csv

[logic2] claude-opus-4-8=23 | gpt-5.6-sol=19 -> 17 ids in intersection | Saved to step1_filter2_logic2.csv


/kaggle/working/step1_filter2_logic2.csv

[reference1] claude-opus-4-8=33 | gpt-5.6-sol=28 -> 24 ids in intersection | Saved to step1_filter2_reference1.csv


/kaggle/working/step1_filter2_reference1.csv